In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import signal
from scipy.signal import butter, filtfilt
from pyedflib import EdfReader
from pathlib import Path
import os
import cv2
import shutil
from sklearn.model_selection import train_test_split
from tqdm import tqdm # Adicionado para uma barra de progresso

# --- Funções de Processamento de Dados (Mantidas) ---
def bandpass_filter(data, lowcut, highcut, fs, order=4):
    nyquist = 0.5 * fs
    low = lowcut / nyquist
    high = highcut / nyquist
    b, a = butter(order, [low, high], btype='band')
    return filtfilt(b, a, data)

def notch_filter(data, notch_freq, fs, quality=30):
    # Correção para scipy >= 1.2.0. Se usar versão antiga, rever esta função.
    b, a = signal.iirnotch(notch_freq, quality, fs=fs)
    return filtfilt(b, a, data)

def compute_individual_epoch_stft(data, events, event_type, tmin, tmax, fs,
                                  nperseg, noverlap, nfft):
    event_samples = [e[0] for e in events if e[2] == event_type]
    individual_stft_data = []
    for i, sample in enumerate(event_samples):
        start = sample + int(tmin * fs)
        end = sample + int(tmax * fs)
        epoch = data[start:end]
        if len(epoch) < nperseg:
            continue
        f, t, Zxx = signal.stft(epoch, fs=fs, window='hann',
                                nperseg=nperseg, noverlap=noverlap, nfft=nfft)
        individual_stft_data.append({'epoch_idx': i, 'frequencies': f, 'times': t, 'magnitude': np.abs(Zxx)})
    return individual_stft_data

def save_spectrograms_as_image(c3_magnitude, c4_magnitude, output_path):
    # Se uma das matrizes estiver vazia, não salve a imagem.
    if c3_magnitude.size == 0 or c4_magnitude.size == 0:
        return None # Retorna None para indicar falha

    log_c3 = np.log1p(c3_magnitude)
    log_c4 = np.log1p(c4_magnitude)

    norm_c3 = cv2.normalize(log_c3, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    norm_c4 = cv2.normalize(log_c4, None, 0, 255, cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    blue_channel = np.zeros_like(norm_c3)
    bgr_image = cv2.merge([blue_channel, norm_c4, norm_c3])
    
    cv2.imwrite(output_path, bgr_image)
    return bgr_image # Retorna a imagem salva para possível visualização

# --- Configurações Globais ---
pasta_raiz_sujeitos = Path('data/physionet_data') 
start_subject = 1
end_subject = 100
registros = ['R04', 'R08', 'R12']
event_dict = {'T0': 'rest', 'T1': 'left', 'T2': 'right'}
epoch_duration_sec = 4
fs = 160
window_durations_ms = [500]

base_output_data_dir = f"eeg_spectrograms_yolo_{window_durations_ms[0]}ms"

if os.path.exists(base_output_data_dir):
    print(f"Limpando diretório existente: {base_output_data_dir}")
    shutil.rmtree(base_output_data_dir)

all_subject_ids = [f'S{i:03d}' for i in range(start_subject, end_subject + 1)]
train_subjects, val_subjects = train_test_split(all_subject_ids, test_size=0.2, random_state=42)

for split in ['train', 'val']:
    for class_name in event_dict.values():
        os.makedirs(os.path.join(base_output_data_dir, split, class_name), exist_ok=True)

print("\nIniciando a geração de spectrograms para o formato YOLO...")

# --- ### NOVO PARA VISUALIZAÇÃO ### ---
# Lista para armazenar as primeiras imagens e seus títulos para plotagem posterior
images_to_display = []
num_images_to_show = 5
# ------------------------------------

# --- Loop Principal para Processamento ---
for current_window_ms in window_durations_ms:
    current_window_s = current_window_ms / 1000.0
    nperseg = int(fs * current_window_s)
    noverlap = int(nperseg * 0.75) 
    nfft = nperseg * 2
    nperseg = max(1, nperseg)
    noverlap = min(noverlap, nperseg - 1)

    print(f"\n--- Processando com Janela de {current_window_ms}ms (nperseg={nperseg}, noverlap={noverlap}) ---")

    # Usando tqdm para uma barra de progresso visual
    for subject_id in tqdm(all_subject_ids, desc="Processando Sujeitos"):
        subject_folder_path = pasta_raiz_sujeitos / subject_id
        if not subject_folder_path.is_dir():
            continue

        split_folder = 'train' if subject_id in train_subjects else 'val'

        for registro in registros:
            edf_file = next(subject_folder_path.glob(f'*{registro}.edf'), None)
            if not edf_file:
                continue
            
            try:
                reader = EdfReader(str(edf_file))
                signal_labels = reader.getSignalLabels()
                c3_idx = signal_labels.index('C3..')
                c4_idx = signal_labels.index('C4..')
                c3_signal = reader.readSignal(c3_idx)
                c4_signal = reader.readSignal(c4_idx)
                annotations_onset, _, annotations_description = reader.readAnnotations()
                reader.close()
                
                events = [[int(onset * fs), 0, desc] for onset, desc in zip(annotations_onset, annotations_description)]
                c3_filtered = notch_filter(bandpass_filter(c3_signal, 0.5, 40, fs), 60, fs)
                c4_filtered = notch_filter(bandpass_filter(c4_signal, 0.5, 40, fs), 60, fs)
                
                conditions = ['T0', 'T1', 'T2']
                for cond_key in conditions:
                    class_name = event_dict[cond_key]
                    
                    c3_epoch_data = compute_individual_epoch_stft(c3_filtered, events, cond_key, 0, epoch_duration_sec, fs, nperseg, noverlap, nfft)
                    c4_epoch_data = compute_individual_epoch_stft(c4_filtered, events, cond_key, 0, epoch_duration_sec, fs, nperseg, noverlap, nfft)

                    for j in range(len(c3_epoch_data)):
                        Zxx_c3 = c3_epoch_data[j]['magnitude']
                        Zxx_c4 = c4_epoch_data[j]['magnitude']
                        
                        if Zxx_c3 is not None and Zxx_c4 is not None:
                            epoch_idx = c3_epoch_data[j]['epoch_idx']
                            base_filename = f"{subject_id}_{edf_file.stem.replace('.', '_')}_epoch_{epoch_idx+1}.png"
                            output_path = os.path.join(base_output_data_dir, split_folder, class_name, base_filename)
                            
                            # A função agora retorna a imagem BGR se for salva com sucesso
                            saved_image = save_spectrograms_as_image(Zxx_c3, Zxx_c4, output_path)

                            # --- ### NOVO PARA VISUALIZAÇÃO ### ---
                            # Se a imagem foi salva e ainda não temos 5 imagens, guarde-a para exibição
                            if saved_image is not None and len(images_to_display) < num_images_to_show:
                                # Converte de BGR (OpenCV) para RGB (Matplotlib)
                                rgb_image = cv2.cvtColor(saved_image, cv2.COLOR_BGR2RGB)
                                title = f"Classe: {class_name}\nSujeito: {subject_id}"
                                images_to_display.append({'image': rgb_image, 'title': title})
                            # ------------------------------------
            except Exception as e:
                # print(f"Erro ao processar {edf_file.name}: {e}") # Descomente para depurar
                if 'reader' in locals() and reader.isOpen():
                    reader.close()
                continue

print(f"\nProcessamento concluído. Dataset para YOLO salvo em: '{base_output_data_dir}'")


# --- ### NOVO PARA VISUALIZAÇÃO ### ---
# Ao final de todo o processamento, exibe as imagens coletadas
if images_to_display:
    print("\nExibindo 5 imagens de amostra geradas...")
    fig, axes = plt.subplots(1, num_images_to_show, figsize=(20, 4))
    for i, item in enumerate(images_to_display):
        ax = axes[i]
        ax.imshow(item['image'])
        ax.set_title(item['title'])
        ax.axis('off') # Remove os eixos x e y para uma visualização mais limpa
    
    plt.tight_layout()
    plt.show()
else:
    print("\nNenhuma imagem foi gerada para exibição.")
# ------------------------------------

Total de sujeitos: 100
Sujeitos de Treino (80): ['S056', 'S089', 'S027', 'S043', 'S070']...
Sujeitos de Validação (20): ['S084', 'S054', 'S071', 'S046', 'S045']...

Iniciando a geração de spectrograms para o formato YOLO...

--- Processando com Janela de 1000ms (nperseg=160, noverlap=120) ---
  Processando Sujeito: S001
  Processando Sujeito: S002
  Processando Sujeito: S003
  Processando Sujeito: S004
  Processando Sujeito: S005
  Processando Sujeito: S006
  Processando Sujeito: S007
  Processando Sujeito: S008
  Processando Sujeito: S009
  Processando Sujeito: S010
  Processando Sujeito: S011
  Processando Sujeito: S012
  Processando Sujeito: S013
  Processando Sujeito: S014
  Processando Sujeito: S015
  Processando Sujeito: S016
  Processando Sujeito: S017
  Processando Sujeito: S018
  Processando Sujeito: S019
  Processando Sujeito: S020
  Processando Sujeito: S021
  Processando Sujeito: S022
  Processando Sujeito: S023
  Processando Sujeito: S024
  Processando Sujeito: S025
  Proc